# Building an MLP with nn.Module
So far we've defined weights manually. In practice, you use `nn.Module` — PyTorch's base class for all neural networks.

In Scribe, every component (the encoder, the conductor, the worker) is an `nn.Module`. Learning this pattern now is essential.

## 1. Your First nn.Module
Defining a model means:
1. Inherit from `nn.Module`
2. Define layers in `__init__`
3. Describe the forward pass in `forward`

In [ ]:
import torch
import torch.nn as nn

class SimpleLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)   # 1 input, 1 output

    def forward(self, x):
        return self.linear(x)

model = SimpleLinear()
print(model)
print("Parameters:", list(model.parameters()))


In [ ]:
# TODO: Define a class called TwoLayerNet with:
#   - a first linear layer: 1 input -> 16 hidden units
#   - a ReLU activation
#   - a second linear layer: 16 hidden -> 1 output
# Then instantiate it and print the model.
# YOUR CODE HERE

class TwoLayerNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(1, 16)
        self.relu = nn.ReLU()
        self.l2 = nn.Linear(16, 1)

    def forward(self, x):
        x = self.l1(x)
        x = self.relu(x)
        x = self.l2(x)
        return x

net = TwoLayerNet()
print(net)


## 2. Training the MLP on a Sine Wave
Let's train our two-layer network to approximate a sine wave — a classic sanity check.

In [ ]:
import matplotlib.pyplot as plt
torch.manual_seed(0)

# Dataset: y = sin(x) with some noise
X = torch.linspace(-3.14, 3.14, 200).unsqueeze(1)
y = torch.sin(X) + 0.05 * torch.randn_like(X)

plt.plot(X, y, label="sin(x) + noise")
plt.title("Training data"); plt.legend(); plt.show()


In [ ]:
# Define the model (use your TwoLayerNet if you wrote it, or this one)
class MLP(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x):
        return self.net(x)

model = MLP(hidden=32)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

losses = []
for epoch in range(500):
    y_pred = model(X)
    loss = loss_fn(y_pred, y)
    losses.append(loss.item())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f"Final loss: {losses[-1]:.5f}")


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(losses); plt.title("Training loss"); plt.xlabel("Epoch")

plt.subplot(1, 2, 2)
with torch.no_grad():
    y_hat = model(X)
plt.plot(X, y, label="true"); plt.plot(X, y_hat, label="predicted")
plt.legend(); plt.title("Model fit")
plt.tight_layout(); plt.show()

# TODO: Change hidden=32 to hidden=4. Does the model still fit the sine wave?
#       Then try hidden=128. Write your observations as a comment.
# YOUR CODE HERE

lossfn = nn.MSELoss()

model4 = MLP(hidden=4)
opt4 = torch.optim.Adam(model4.parameters(), lr=0.01)

for i in range(500):
    pred4 = model4(X)
    l4 = lossfn(pred4, y)
    opt4.zero_grad()
    l4.backward()
    opt4.step()

model128 = MLP(hidden=128)
opt128 = torch.optim.Adam(model128.parameters(), lr=0.01)

for i in range(500):
    pred128 = model128(X)
    l128 = lossfn(pred128, y)
    opt128.zero_grad()
    l128.backward()
    opt128.step()


## 3. Saving & Loading a Model
You'll need this every time you want to reuse a trained model.

In [ ]:
# Save
torch.save(model.state_dict(), "mlp_sine.pth")
print("Model saved.")

# Load
model2 = MLP(hidden=32)
model2.load_state_dict(torch.load("mlp_sine.pth"))
model2.eval()
print("Model loaded.")

# TODO: Verify the loaded model gives the same output as the original on X[:5]
# Hint: use torch.allclose()
# YOUR CODE HERE

out1 = model(X[:5])
out2 = model2(X[:5])
match = torch.allclose(out1, out2)
print(match)


## 4. Connecting to Scribe
In Scribe, the encoder will be an `nn.Module` that takes a stroke tensor of shape `(batch, seq_len, 5)` and produces a latent vector `z`. The decoder will take `z` and produce strokes back. Both follow the exact same `__init__ / forward` pattern you just learned.

In [ ]:
# A sketch of what the Scribe encoder will look like (Week 3)
# Don't worry about understanding every line — just notice the structure.

class SketchEncoder(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=256, latent_dim=128):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc_mu  = nn.Linear(hidden_dim * 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim * 2, latent_dim)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h = torch.cat([h[0], h[1]], dim=-1)   # concatenate both directions
        mu     = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

print("SketchEncoder defined — we'll implement this properly in Week 3.")


## Done!
You can now define, train, save, and load PyTorch models.
Next: **assignment/assignment.ipynb** — train a classifier on FashionMNIST.